In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedShuffleSplit, TimeSeriesSplit
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from scipy.sparse import hstack
from sklearn.inspection import permutation_importance
from sklearn.metrics import root_mean_squared_error
from catboost import CatBoostRegressor


In [ ]:
RANDOM_STATE = 42

Повторяю дейтсвия для работы с небольшим датасетом:

In [ ]:

df_train = pd.read_csv("../train.csv")
_y = (df_train['deal_probability'].round(2) * 100).astype(int)
splitter = StratifiedShuffleSplit(n_splits=1, test_size=300000, random_state=42)
rest_idx, small_idx = next(splitter.split(df_train, _y))
df_small = df_train.iloc[small_idx].reset_index(drop=True)
df_small['activation_date'] = pd.to_datetime(df_small['activation_date'])
df_small = df_small.sort_values(by='activation_date').reset_index(drop=True)

length = len(df_small.index)
train_end_id = int(0.8 * length)
valid_end_id = int(0.9 * length)

df_small_train = df_small.iloc[:train_end_id, :]
df_small_valid = df_small.iloc[train_end_id:valid_end_id, :]
df_small_test = df_small.iloc[valid_end_id:, :]

## TF-IDF на топ 2000 слов из признака title

In [ ]:
def gradient_tfidf_preprocessing_fit_transform(df_train):
    df = df_train.copy()

    num_imputer = SimpleImputer(strategy='median')
    df['price'] = num_imputer.fit_transform(df[['price']])

    cat_imputer = SimpleImputer(strategy='constant', fill_value='missing')
    cat_cols = ['param_1', 'image_top_1']
    df[cat_cols] = cat_imputer.fit_transform(df[cat_cols])

    df['activation_date'] = pd.to_datetime(df['activation_date'])
    df['activation_day_of_week'] = df['activation_date'].dt.day_of_week
    df['activation_day_of_week'] = df['activation_day_of_week'].astype(str)
    df['image_top_1'] = df['image_top_1'].astype(str)

    feature_names = ['region', 'city', 'parent_category_name', 'category_name', 'param_1', 'activation_day_of_week', 'user_type', 'image_top_1', 'price']
    df['title'] = df['title'].astype(str)
    russian_stopwords = stopwords.words('russian')
    tfidf = TfidfVectorizer(max_features=2000, stop_words=russian_stopwords)
    df_tfidf_sparse = tfidf.fit_transform(df['title'])
    tfidf_cols = [f'tfidf_{i}' for i in range(df_tfidf_sparse.shape[1])]
    df_tfidf = pd.DataFrame.sparse.from_spmatrix(df_tfidf_sparse, columns=tfidf_cols, index=df.index)
    
    X_train = pd.concat([df[feature_names], df_tfidf], axis=1)    
    y_train = df['deal_probability']

    return X_train, y_train, num_imputer, cat_imputer, feature_names, tfidf


def gradient_tfidf_preprocessing_transform(df_valid, num_imputer, cat_imputer, tfidf):
    df = df_valid.copy()

    df['price'] = num_imputer.transform(df[['price']])

    cat_cols = ['param_1', 'image_top_1']
    df[cat_cols] = cat_imputer.transform(df[cat_cols])

    df['activation_date'] = pd.to_datetime(df['activation_date'])
    df['activation_day_of_week'] = df['activation_date'].dt.day_of_week
    df['activation_day_of_week'] = df['activation_day_of_week'].astype(str)
    df['image_top_1'] = df['image_top_1'].astype(str)

    feature_names = ['region', 'city', 'parent_category_name', 'category_name', 'param_1', 'activation_day_of_week', 'user_type', 'image_top_1', 'price']

    df['title'] = df['title'].astype(str)
    df_tfidf_sparse = tfidf.transform(df['title'])
    
    tfidf_cols = [f'tfidf_{i}' for i in range(df_tfidf_sparse.shape[1])]
    df_tfidf = pd.DataFrame.sparse.from_spmatrix(df_tfidf_sparse, columns=tfidf_cols, index=df.index)
    X_valid = pd.concat([df[feature_names], df_tfidf], axis=1)
    
    if 'deal_probability' in df.columns:
        y_valid = df['deal_probability']
    else:
        y_valid = None

    return X_valid, y_valid

In [ ]:
%%time
print(f"CatBoost + TF-IDF")
print()
kf = KFold(n_splits=5, shuffle=True, random_state=42)
scores = []
    

for train_idx, val_idx in kf.split(df_small_train):
    df_tr, df_val = df_small_train.iloc[train_idx], df_small_train.iloc[val_idx]
    n = len(df_tr)
    cut = int(n * 0.9)
    df_tr_fit, df_tr_es = df_tr.iloc[:cut], df_tr.iloc[cut:]
    
    

    X_tr, y_tr, num_imputer, cat_imputer, feature_names, tfidf = gradient_tfidf_preprocessing_fit_transform(df_tr_fit)
    X_vl, y_vl = gradient_tfidf_preprocessing_transform(df_val, num_imputer, cat_imputer, tfidf)
    X_val_eval, y_val_eval = gradient_tfidf_preprocessing_transform(df_tr_es, num_imputer, cat_imputer, tfidf)

    model = CatBoostRegressor(learning_rate=0.1, cat_features=['region', 'city', 
    'parent_category_name', 'category_name', 'param_1', 'activation_day_of_week', 'user_type', 'image_top_1'], verbose=False)
    model.fit(X_tr, y_tr, eval_set=(X_val_eval, y_val_eval))

    preds = model.predict(X_vl)
    fold_rmse = root_mean_squared_error(y_vl, preds)
    scores.append(fold_rmse)
    print(f"Получили RMSE на фолде: {fold_rmse}")
print(f"Среднее RMSE: {np.mean(scores)}")        

CatBoost + TF-IDF

Получили RMSE на фолде: 0.2301890079865448
Получили RMSE на фолде: 0.2300603747777713
Получили RMSE на фолде: 0.2265734171045529
Получили RMSE на фолде: 0.22828460935256847
Получили RMSE на фолде: 0.22695603912044796
Среднее RMSE: 0.22841268966837708
CPU times: user 35min 28s, sys: 1min 38s, total: 37min 6s
Wall time: 4min 23s


## Теперь предварительно сделаем лемматизацию

In [ ]:
m = Mystem()
russian_stopwords = stopwords.words('russian')
TEXT_SEPARATOR = " xseparatorx " 


def gradient_lemm_preprocessing_fit_transform(df_train):
    df = df_train.copy()

    num_imputer = SimpleImputer(strategy='median')
    df['price'] = num_imputer.fit_transform(df[['price']])

    cat_imputer = SimpleImputer(strategy='constant', fill_value='missing')
    cat_cols = ['param_1', 'image_top_1']
    df[cat_cols] = cat_imputer.fit_transform(df[cat_cols])

    df['activation_date'] = pd.to_datetime(df['activation_date'])
    df['activation_day_of_week'] = df['activation_date'].dt.day_of_week
    df['activation_day_of_week'] = df['activation_day_of_week'].astype(str)
    df['image_top_1'] = df['image_top_1'].astype(str)

    feature_names = ['region', 'city', 'parent_category_name', 'category_name', 'param_1', 'activation_day_of_week', 'user_type', 'image_top_1', 'price']
    df['title'] = df['title'].astype(str)

    all_text = TEXT_SEPARATOR.join(df['title'].fillna('').astype(str)).lower()
    all_text_cleaned = re.sub(r'[^а-яёa-z\s]', ' ', all_text)
    lemmatized_all = m.lemmatize(all_text_cleaned)
    all_text_lemmatized = "".join(lemmatized_all)
    df['cleaned_title'] = [text.strip() for text in all_text_lemmatized.split(TEXT_SEPARATOR)]

    tfidf = TfidfVectorizer(max_features=2000, stop_words=russian_stopwords)
    df_tfidf_sparse = tfidf.fit_transform(df['cleaned_title'])
    tfidf_cols = [f'tfidf_{i}' for i in range(df_tfidf_sparse.shape[1])]
    df_tfidf = pd.DataFrame.sparse.from_spmatrix(df_tfidf_sparse, columns=tfidf_cols, index=df.index)
    
    X_train = pd.concat([df[feature_names], df_tfidf], axis=1)    
    y_train = df['deal_probability']

    return X_train, y_train, num_imputer, cat_imputer, feature_names, tfidf


def gradient_lemm_preprocessing_transform(df_valid, num_imputer, cat_imputer, tfidf):
    df = df_valid.copy()

    df['price'] = num_imputer.transform(df[['price']])

    cat_cols = ['param_1', 'image_top_1']
    df[cat_cols] = cat_imputer.transform(df[cat_cols])

    df['activation_date'] = pd.to_datetime(df['activation_date'])
    df['activation_day_of_week'] = df['activation_date'].dt.day_of_week
    df['activation_day_of_week'] = df['activation_day_of_week'].astype(str)
    df['image_top_1'] = df['image_top_1'].astype(str)

    feature_names = ['region', 'city', 'parent_category_name', 'category_name', 'param_1', 'activation_day_of_week', 'user_type', 'image_top_1', 'price']

    df['title'] = df['title'].astype(str)

    all_text = TEXT_SEPARATOR.join(df['title'].fillna('').astype(str)).lower()
    all_text_cleaned = re.sub(r'[^а-яёa-z\s]', ' ', all_text)
    lemmatized_all = m.lemmatize(all_text_cleaned)
    all_text_lemmatized = "".join(lemmatized_all)
    df['cleaned_title'] = [text.strip() for text in all_text_lemmatized.split(TEXT_SEPARATOR)]

    df_tfidf_sparse = tfidf.transform(df['cleaned_title'])        
    tfidf_cols = [f'tfidf_{i}' for i in range(df_tfidf_sparse.shape[1])]
    df_tfidf = pd.DataFrame.sparse.from_spmatrix(df_tfidf_sparse, columns=tfidf_cols, index=df.index)    
    X_valid = pd.concat([df[feature_names], df_tfidf], axis=1)
    
    if 'deal_probability' in df.columns:
        y_valid = df['deal_probability']
    else:
        y_valid = None

    return X_valid, y_valid

In [ ]:
%%time
print(f"CatBoost + TF-IDF + lemmatization")
print()
kf = KFold(n_splits=5, shuffle=True, random_state=42)
scores = []
    
for train_idx, val_idx in kf.split(df_small_train):
    df_tr, df_val = df_small_train.iloc[train_idx], df_small_train.iloc[val_idx]

    X_tr, y_tr, num_imputer, cat_imputer, feature_names, tfidf = gradient_lemm_preprocessing_fit_transform(df_tr)
    X_vl, y_vl = gradient_lemm_preprocessing_transform(df_val, num_imputer, cat_imputer, tfidf)
    X_val_eval, y_val_eval = gradient_lemm_preprocessing_transform(df_small_valid, num_imputer, cat_imputer, tfidf)

    model = CatBoostRegressor(learning_rate=0.1, cat_features=['region', 'city', 
    'parent_category_name', 'category_name', 'param_1', 'activation_day_of_week', 'user_type', 'image_top_1'], verbose=False)
    model.fit(X_tr, y_tr, eval_set=(X_val_eval, y_val_eval))

    preds = model.predict(X_vl)
    fold_rmse = root_mean_squared_error(y_vl, preds)
    scores.append(fold_rmse)
    print(f"Получили RMSE на фолде: {fold_rmse}")
print(f"Среднее RMSE: {np.mean(scores)}")        

CatBoost + TF-IDF + lemmatization

Получили RMSE на фолде: 0.22978641755584553
Получили RMSE на фолде: 0.22959491428739023
Получили RMSE на фолде: 0.226236319718684
Получили RMSE на фолде: 0.22797141499658163
Получили RMSE на фолде: 0.22666270508785405
Среднее RMSE: 0.22805035432927107
CPU times: user 59min 18s, sys: 2min 29s, total: 1h 1min 47s
Wall time: 22min 52s


RMSE улучился, но не сильно

Поскольку CatBoost умеет работать с текстовыми признаками сам, посмотрим, получится ли у него лучше

## Catboost + text_features

In [ ]:
def gradient_text_preprocessing_fit_transform(df_train):
    df = df_train.copy()

    num_imputer = SimpleImputer(strategy='median')
    df['price'] = num_imputer.fit_transform(df[['price']])

    cat_imputer = SimpleImputer(strategy='constant', fill_value='missing')
    cat_cols = ['param_1', 'image_top_1']
    df[cat_cols] = cat_imputer.fit_transform(df[cat_cols])

    df['activation_date'] = pd.to_datetime(df['activation_date'])
    df['activation_day_of_week'] = df['activation_date'].dt.day_of_week
    df['activation_day_of_week'] = df['activation_day_of_week'].astype(str)
    df['image_top_1'] = df['image_top_1'].astype(str)
    df['title'] = df['title'].astype(str)
    df['description'] = df['description'].astype(str)


    feature_names = ['region', 'city', 'parent_category_name', 'category_name', 'param_1', 'activation_day_of_week', 'user_type', 'image_top_1', 'price', 'title', 'description']
    
    X_train = df[feature_names]
    y_train = df['deal_probability']

    return X_train, y_train, num_imputer, cat_imputer, feature_names


def gradient_text_preprocessing_transform(df_valid, num_imputer, cat_imputer):
    df = df_valid.copy()

    df['price'] = num_imputer.transform(df[['price']])

    cat_cols = ['param_1', 'image_top_1']
    df[cat_cols] = cat_imputer.transform(df[cat_cols])

    df['activation_date'] = pd.to_datetime(df['activation_date'])
    df['activation_day_of_week'] = df['activation_date'].dt.day_of_week
    df['activation_day_of_week'] = df['activation_day_of_week'].astype(str)
    df['image_top_1'] = df['image_top_1'].astype(str)
    df['title'] = df['title'].astype(str)
    df['description'] = df['description'].astype(str)



    feature_names = ['region', 'city', 'parent_category_name', 'category_name', 'param_1', 'activation_day_of_week', 'user_type', 'image_top_1', 'price', 'title', 'description']
    
    X_valid = df[feature_names]
    
    if 'deal_probability' in df.columns:
        y_valid = df['deal_probability']
    else:
        y_valid = None

    return X_valid, y_valid

In [ ]:
%%time
print(f"CatBoost + text_features")
print()
kf = KFold(n_splits=5, shuffle=True, random_state=42)
scores = []
    
for train_idx, val_idx in kf.split(df_small_train):
    df_tr, df_val = df_small_train.iloc[train_idx], df_small_train.iloc[val_idx]

    X_tr, y_tr, num_imputer, cat_imputer, feature_names = gradient_text_preprocessing_fit_transform(df_tr)
    X_vl, y_vl = gradient_text_preprocessing_transform(df_val, num_imputer, cat_imputer)
    X_val_eval, y_val_eval = gradient_text_preprocessing_transform(df_small_valid, num_imputer, cat_imputer)

    model = CatBoostRegressor(learning_rate=0.1, cat_features=['region', 'city', 
    'parent_category_name', 'category_name', 'param_1', 'activation_day_of_week', 'user_type', 'image_top_1'], text_features=['title'], verbose=False)
    model.fit(X_tr, y_tr, eval_set=(X_val_eval, y_val_eval))

    preds = model.predict(X_vl)
    fold_rmse = root_mean_squared_error(y_vl, preds)
    scores.append(fold_rmse)
    print(f"Получили RMSE на фолде: {fold_rmse}")
print(f"Среднее RMSE: {np.mean(scores)}")        

CatBoost + text_features

Получили RMSE на фолде: 0.22945417019210554
Получили RMSE на фолде: 0.22960511943555822
Получили RMSE на фолде: 0.22587565904623932
Получили RMSE на фолде: 0.22771960204909902
Получили RMSE на фолде: 0.22657271365984025
Среднее RMSE: 0.22784545287656846
CPU times: user 1h 8min 11s, sys: 2min 18s, total: 1h 10min 30s
Wall time: 8min 2s


Как видим, CatBoost справился лучше, еще и намного быстрее (почти в 3 раза)

Раз уж такое дело, то скормим катбусту все текстовые признаки

## CatBoost + text_features (title, description)

In [ ]:
%%time
print(f"CatBoost + text_features (title+description)")
print()
kf = KFold(n_splits=5, shuffle=True, random_state=42)
scores = []
    
for train_idx, val_idx in kf.split(df_small_train):
    df_tr, df_val = df_small_train.iloc[train_idx], df_small_train.iloc[val_idx]

    X_tr, y_tr, num_imputer, cat_imputer, feature_names = gradient_text_preprocessing_fit_transform(df_tr)
    X_vl, y_vl = gradient_text_preprocessing_transform(df_val, num_imputer, cat_imputer)
    X_val_eval, y_val_eval = gradient_text_preprocessing_transform(df_small_valid, num_imputer, cat_imputer)

    model = CatBoostRegressor(learning_rate=0.1, cat_features=['region', 'city', 
    'parent_category_name', 'category_name', 'param_1', 'activation_day_of_week', 'user_type', 'image_top_1'], text_features=['title', 'description'], verbose=False)
    model.fit(X_tr, y_tr, eval_set=(X_val_eval, y_val_eval))

    preds = model.predict(X_vl)
    fold_rmse = root_mean_squared_error(y_vl, preds)
    scores.append(fold_rmse)
    print(f"Получили RMSE на фолде: {fold_rmse}")
print(f"Среднее RMSE: {np.mean(scores)}")        

CatBoost + text_features (title+description)

Получили RMSE на фолде: 0.22894727233025758
Получили RMSE на фолде: 0.2286444151540445
Получили RMSE на фолде: 0.22538120358474084
Получили RMSE на фолде: 0.22725984049129036
Получили RMSE на фолде: 0.22586283898286558
Среднее RMSE: 0.22721911410863976
CPU times: user 1h 57min 15s, sys: 3min 8s, total: 2h 23s
Wall time: 13min 27s


Итого на валидации получили RMSE = 0.227